# 00 Data Processing


In [1]:
%load_ext autoreload
%autoreload 2

import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from numpy import linalg as la

from config import (
    DATA_DIR,
    MOLECULE_DIR,
    PARAMETERS_DIR,
    RAW_MATRICES_DIR,
    PROCESSED_DATAFRAMES_DIR
)
from notebook_utils.general import complex_matrix, read_jsonl

### Statevector Dataset

In [2]:
SV_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "SV_hamiltonian"
SV_SPIN_DIR = RAW_MATRICES_DIR / "SV_spin"
PYSCF_CASCI_ENERGIES_PATH = RAW_MATRICES_DIR / "pyscf_casci_energies.jsonl"
def load_sv_hamiltonian_records(path):
    records = []

    for file in sorted(path.glob("*.jsonl")):
        for row in read_jsonl(file):
            H = complex_matrix(row["h_matrix_real"], row["h_matrix_imag"])
            S = complex_matrix(row["s_matrix_real"], row["s_matrix_imag"])

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "H": H,
                "S": S,
                "qse_dim": H.shape[0],
            })

    return pd.DataFrame(records)


def load_spin_records(path):
    records = []

    for file in sorted(path.glob("*.jsonl")):
        for row in read_jsonl(file):
            S2 = complex_matrix(row["z_matrix_real"], row["z_matrix_imag"])

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "S2": S2
            })

    return pd.DataFrame(records)

def load_pyscf_casci_energy_records(path):
    records = []

    for row in read_jsonl(path):
        records.append({
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "spin_type": row["spin_type"],
            "pyscf_casci_energies": row["casci_energies"],
            "pyscf_casci_pvec": row["casci_pvec"],
        })

    return pd.DataFrame(records)


df_sv_hamiltonian = load_sv_hamiltonian_records(SV_HAMILTONIAN_DIR)
df_sv_spin = load_spin_records(SV_SPIN_DIR)
df_pyscf_casci = load_pyscf_casci_energy_records(PYSCF_CASCI_ENERGIES_PATH)


In [3]:
# Compress the PYSCF Triplet energies
def average_triplet_sector_energies(casci_energies):
    average_energies = []
    sector_diffs = []

    for nth_energies in casci_energies:
        energies = list(nth_energies.values())
        average_energies.append(float(np.mean(energies)))
        sector_diffs.append(float(np.ptp(energies)))

    return average_energies, sector_diffs


new_casci_energies = []
triplet_sector_diffs = []

for _, row in df_pyscf_casci.iterrows():
    if row["spin_type"] == "triplet_all":
        energies, sector_diffs = average_triplet_sector_energies(
            row["pyscf_casci_energies"]
        )
        triplet_sector_diffs.extend(sector_diffs)
    else:
        energies = row["pyscf_casci_energies"]

    new_casci_energies.append(energies)


df_pyscf_casci["pyscf_casci_energies"] = new_casci_energies

max_triplet_sector_diff = max(triplet_sector_diffs, default=0.0)
print(f"Maximum PYSCF triplet sector energy difference: {max_triplet_sector_diff:.12e} Ha")


Maximum PYSCF triplet sector energy difference: 1.003218130791e-08 Ha


In [4]:
df_sv = df_sv_hamiltonian.merge(
    df_sv_spin,
    on=["molecule", "active_space", "ansatz", "expansion"],
    how="left",
)

df_sv = df_sv.merge(
    df_pyscf_casci,
    left_on=["molecule", "active_space", "expansion"],
    right_on=["molecule", "active_space", "spin_type"],
    how="left",
).drop(columns="spin_type")

df_sv = df_sv.sort_values(
    ["molecule", "active_space", "ansatz", "expansion"]
).reset_index(drop=True)

In [5]:
required_columns = ["H", "S", "S2", "pyscf_casci_energies", "pyscf_casci_pvec"]

df_sv_complete = df_sv.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SV_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "sv_qse_data.pkl"

df_sv_complete.to_pickle(SV_DATAFRAME_PATH)

df_sv_complete


,molecule,active_space,ansatz,expansion,H,S,qse_dim,S2,pyscf_casci_energies,pyscf_casci_pvec
0,Acetamide,2e2o,1UpCCGSDSinglet,singlet,"[[(-802.5663414572622+0j), (-4.706692634862637...","[[(3.9097041888243123+0j), (0.0229245233142809...",4,"[[(1.27675647831893e-14+0j), (-4.3368086899420...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098..."
1,Acetamide,2e2o,1UpCCGSDSinglet,triplet_all,"[[(-0.008069504761204022+0j), 0j, 0j, (0.19305...","[[(3.934912814912428e-05+0j), 0j, 0j, (-0.0009...",12,"[[(7.869825629530647e-05+0j), 0j, 0j, (-0.0018...",[-205.07455054622508],"{'0_0': [-4.001328838134542e-16, 0, -0.7071067..."
2,Acetamide,2e2o,UCCGSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098..."
3,Acetamide,2e2o,UCCGSD,triplet_all,"[[(-0.008087395184562356+0j), 0j, 0j, (0.19324...","[[(3.943636678180318e-05+0j), 0j, 0j, (-0.0009...",12,"[[(7.887273356062263e-05+0j), 0j, 0j, (-0.0018...",[-205.07455054622508],"{'0_0': [-4.001328838134542e-16, 0, -0.7071067..."
4,Acetamide,2e2o,UCCSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098..."
...,...,...,...,...,...,...,...,...,...,...
1539,Uracil,6e5o,UCCSDSinglet,triplet_all,"[[(-5.3717030823463574e-11+0j), 0j, 0j, (-5.73...","[[(1.323663401109343e-13+0j), 0j, 0j, (1.41031...",75,"[[(2.9176661087149114e-13+0j), 0j, 0j, 0j, 0j,...","[-406.93960384512866, -406.85734583999374, -40...","{'0_0': [2.5283724731332295e-16, 0, -2.7969232..."
1540,Uracil,6e6o,1UpCCGSDSinglet,singlet,"[[(-1628.297155991536+0j), 0j, 0j, 0j, 0j, 0j,...","[[(3.9997456147951955+0j), 0j, 0j, 0j, 0j, 0j,...",36,"[[(6.306155684449033e-13+0j), 0j, 0j, 0j, 0j, ...","[-407.12116588854, -406.889567697793, -406.821...","{'0': [0.00023156095438382814, 0, 4.8560227557..."
1541,Uracil,6e6o,UCCGSD,singlet,"[[(-1628.135667965554+0j), 0j, 0j, 0j, 0j, 0j,...","[[(3.9991449002458106+0j), 0j, 0j, 0j, 0j, 0j,...",36,"[[(0.000585094449729357+0j), 0j, 0j, 0j, 0j, 0...","[-407.12116588854, -406.889567697793, -406.821...","{'0': [0.00023156095438382814, 0, 4.8560227557..."
1542,Uracil,6e6o,UCCSD,singlet,"[[(-1628.146797538434+0j), 0j, 0j, 0j, 0j, 0j,...","[[(3.9991745183913965+0j), 0j, 0j, 0j, 0j, 0j,...",36,"[[(4.877871797074555e-07+0j), 0j, 0j, 0j, 0j, ...","[-407.12116588854, -406.889567697793, -406.821...","{'0': [0.00023156095438382814, 0, 4.8560227557..."


### Shots dataset

In [6]:
SHOTS_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "shots_hamiltonian"


def load_shots_hamiltonian_records(path):
    records = []

    for file in sorted(path.glob("*.jsonl")):
        for row in read_jsonl(file):
            H = complex_matrix(row["h_matrix_real"], row["h_matrix_imag"])
            S = complex_matrix(row["s_matrix_real"], row["s_matrix_imag"])

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "sample_key": row["sample_key"],
                "n_shots": row["n_shots"],
                "repeat": row["repeat"],
                "H_shots": H,
                "S_shots": S,
                "qse_dim": H.shape[0],
            })

    return pd.DataFrame(records)


df_shots_hamiltonian = load_shots_hamiltonian_records(SHOTS_HAMILTONIAN_DIR)

df_shots_sv = df_sv_hamiltonian.rename(columns={
    "H": "H_sv",
    "S": "S_sv",
    "qse_dim": "qse_dim_sv",
})

df_shots = df_shots_hamiltonian.merge(
    df_shots_sv,
    on=["molecule", "active_space", "ansatz", "expansion"],
    how="left",
)

df_shots = df_shots.sort_values(
    ["molecule", "active_space", "ansatz", "expansion", "n_shots", "repeat"]
).reset_index(drop=True)


In [7]:
required_columns = ["H_shots", "S_shots", "H_sv", "S_sv"]

df_shots_complete = df_shots.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SHOTS_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "shots_qse_data.pkl"

df_shots_complete.to_pickle(SHOTS_DATAFRAME_PATH)

df_shots_complete


,molecule,active_space,ansatz,expansion,sample_key,n_shots,repeat,H_shots,S_shots,qse_dim,H_sv,S_sv,qse_dim_sv
0,Acetamide,2e2o,UCCSD,singlet,shots_1000_0,1000,0,"[[(-799.5517868114334+0j), (3.8471454421130837...","[[(3.8949999999999996+0j), (-0.018749999999999...",4,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4
1,Acetamide,2e2o,UCCSD,singlet,shots_1000_1,1000,1,"[[(-799.3450014736095+0j), (-18.06551591062977...","[[(3.894+0j), (0.08800000000000004-0.020750000...",4,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4
2,Acetamide,2e2o,UCCSD,singlet,shots_1000_2,1000,2,"[[(-802.215783891827+0j), (5.748233963083942+1...","[[(3.9080000000000004+0j), (-0.028000000000000...",4,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4
3,Acetamide,2e2o,UCCSD,singlet,shots_1000_3,1000,3,"[[(-800.1703772137878+0j), (-2.411631973619889...","[[(3.898+0j), (0.01175000000000001+0.026249999...",4,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4
4,Acetamide,2e2o,UCCSD,singlet,shots_1000_4,1000,4,"[[(-801.3992815429581+0j), (1.2815546057821785...","[[(3.904+0j), (-0.006250000000000016-0.0167500...",4,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67195,Uracil,6e5o,UCCSDSinglet,triplet_all,shots_1000000_5,1000000,5,"[[(-5.3715056097303204e-05+0j), (0.10212737830...","[[(-5.551115123125783e-17+0j), (-0.00025102290...",75,"[[(-5.3717030823463574e-11+0j), 0j, 0j, (-5.73...","[[(1.323663401109343e-13+0j), 0j, 0j, (1.41031...",75
67196,Uracil,6e5o,UCCSDSinglet,triplet_all,shots_1000000_6,1000000,6,"[[(-4.7730582764415885e-05+0j), (0.11068336616...","[[(2.7755575615628914e-17+0j), (-0.00027223611...",75,"[[(-5.3717030823463574e-11+0j), 0j, 0j, (-5.73...","[[(1.323663401109343e-13+0j), 0j, 0j, (1.41031...",75
67197,Uracil,6e5o,UCCSDSinglet,triplet_all,shots_1000000_7,1000000,7,"[[(1.0661799507261094e-05+0j), (0.039500714743...","[[(5.551115123125783e-17+0j), (-9.758073580369...",75,"[[(-5.3717030823463574e-11+0j), 0j, 0j, (-5.73...","[[(1.323663401109343e-13+0j), 0j, 0j, (1.41031...",75
67198,Uracil,6e5o,UCCSDSinglet,triplet_all,shots_1000000_8,1000000,8,"[[(0.00017228524254164768+0j), (-0.16622784707...","[[(2.7755575615628914e-17+0j), (0.000409061272...",75,"[[(-5.3717030823463574e-11+0j), 0j, 0j, (-5.73...","[[(1.323663401109343e-13+0j), 0j, 0j, (1.41031...",75
